In [7]:
import tensorflow as tf

batch_size = 32
img_height = 224
img_width = 224

# Diviser les données en 80% train et 20% pour validation/test
train_val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset',
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

val_test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset',
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

# Diviser les 20% en 10% pour validation et 10% pour test
val_size = int(0.5 * len(val_test_ds))  # 50% de val_test_ds pour validation
test_size = len(val_test_ds) - val_size  # Reste pour test

val_ds = val_test_ds.take(val_size)  # Premier 50% pour validation
test_ds = val_test_ds.skip(val_size)  # Dernier 50% pour test

# Vérifier la répartition
print(f"Nombre de batches dans train_ds: {len(train_val_ds)}")
print(f"Nombre de batches dans val_ds: {len(val_ds)}")
print(f"Nombre de batches dans test_ds: {len(test_ds)}")

Found 11634 files belonging to 3 classes.
Using 9308 files for training.
Found 11634 files belonging to 3 classes.
Using 2326 files for validation.
Nombre de batches dans train_ds: 291
Nombre de batches dans val_ds: 36
Nombre de batches dans test_ds: 37


In [8]:
## chargement dataset maiis
# Diviser les données en 80% train et 20% pour validation/test
train_val_ds_maiis = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset maiis',
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

val_test_ds_maiis = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset maiis',
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

# Diviser les 20% en 10% pour validation et 10% pour test
val_size = int(0.5 * len(val_test_ds_maiis))  # 50% de val_test_ds pour validation
test_size = len(val_test_ds_maiis) - val_size  # Reste pour test

val_ds_maiis = val_test_ds_maiis.take(val_size)  # Premier 50% pour validation
test_ds_maiis = val_test_ds_maiis.skip(val_size)  # Dernier 50% pour test

# Vérifier la répartition
print(f"Nombre de batches dans train_ds: {len(train_val_ds_maiis)}")
print(f"Nombre de batches dans val_ds: {len(val_ds_maiis)}")
print(f"Nombre de batches dans test_ds: {len(test_ds_maiis)}")

Found 11700 files belonging to 3 classes.
Using 9360 files for training.
Found 11700 files belonging to 3 classes.
Using 2340 files for validation.
Nombre de batches dans train_ds: 293
Nombre de batches dans val_ds: 37
Nombre de batches dans test_ds: 37


In [9]:
## chargement dataset maiis
# Diviser les données en 80% train et 20% pour validation/test
train_val_ds_mixte = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset mixte',
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

val_test_ds_mixte = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset mixte',
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

# Diviser les 20% en 10% pour validation et 10% pour test
val_size_mixte = int(0.5 * len(val_test_ds_mixte))  # 50% de val_test_ds pour validation
test_size_mixte = len(val_test_ds_mixte) - val_size_mixte  # Reste pour test

val_ds_mixte = val_test_ds_maiis.take(val_size_mixte)  # Premier 50% pour validation
test_ds_mixte = val_test_ds_maiis.skip(val_size_mixte)  # Dernier 50% pour test

# Vérifier la répartition
print(f"Nombre de batches dans train_ds: {len(train_val_ds_mixte)}")
print(f"Nombre de batches dans val_ds: {len(val_ds_mixte)}")
print(f"Nombre de batches dans test_ds: {len(test_ds_mixte)}")

Found 11700 files belonging to 3 classes.
Using 9360 files for training.
Found 11700 files belonging to 3 classes.
Using 2340 files for validation.
Nombre de batches dans train_ds: 293
Nombre de batches dans val_ds: 37
Nombre de batches dans test_ds: 37


In [10]:
from tensorflow.keras.applications import MobileNetV3Large  # Utiliser MobileNetV3Large ou MobileNetV3Small
from tensorflow.keras import layers, models
from tensorflow.keras.metrics import Precision, Recall
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import time

# Définir les dimensions d'image pour MobileNetV3
img_height, img_width = 224, 224

# Charger le modèle MobileNetV3 pré-entraîné
base_model = MobileNetV3Large(input_shape=(img_height, img_width, 3),
                              include_top=False,
                              weights='imagenet')
base_model.trainable = False  # Geler les poids du modèle pré-entraîné

for layer in base_model.layers[-50:]:
    layer.trainable = True

# Ajouter une couche Global Average Pooling
global_average_layer = layers.GlobalAveragePooling2D()(base_model.output)

# Ajouter des couches fully connected supplémentaires avec Dropout
dense_1 = layers.Dense(1024, activation='relu')(global_average_layer)
dropout_1 = layers.Dropout(0.5)(dense_1)

dense_2 = layers.Dense(512, activation='relu')(dropout_1)
dropout_2 = layers.Dropout(0.5)(dense_2)

dense_3 = layers.Dense(256, activation='relu')(dropout_2)
dropout_3 = layers.Dropout(0.5)(dense_3)

dense_4 = layers.Dense(128, activation='relu')(dropout_3)
dropout_4 = layers.Dropout(0.5)(dense_4)

# Créer les sorties pour chaque nutriment (13 au total)
outputs = []
for nutrient in range(13):
    output = layers.Dense(3, activation='softmax', name=f'nutrient_{nutrient}')(dropout_4)
    outputs.append(output)

# Créer le modèle final avec MobileNetV3 en entrée et les 13 sorties en sortie
model = models.Model(inputs=base_model.input, outputs=outputs)

# Compilation du modèle avec des métriques adaptées pour chaque sortie
metrics = ['accuracy', Precision(name='precision'), Recall(name='recall')]
metrics_list = [metrics] * 13  # Appliquer les métriques à chaque nutriment

model.compile(optimizer='adam',
              loss=['categorical_crossentropy'] * 13,
              metrics=metrics_list)

# Afficher un résumé du modèle pour vérifier les couches et les sorties
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)    │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ rescaling_1 (Rescaling)       │ (None, 224, 224, 3)       │               0 │ input_layer_1[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv (Conv2D)                 │ (None, 112, 112, 16)      │             432 │ rescaling_1[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv_bn (BatchNormalization)  │ (None, 112, 112, 16)      │              64 │ conv[0][0]                 │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_20 (Activation)    │ (None, 112, 112, 16)      │               0 │ conv_bn[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise       │ (None, 112, 112, 16)      │             144 │ activation_20[0][0]        │
│ (DepthwiseConv2D)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_bn    │ (None, 112, 112, 16)      │              64 │ expanded_conv_depthwise[0… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ re_lu_19 (ReLU)               │ (None, 112, 112, 16)      │               0 │ expanded_conv_depthwise_b… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project         │ (None, 112, 112, 16)      │             256 │ re_lu_19[0][0]             │
│ (Conv2D)                      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project_bn      │ (None, 112, 112, 16)      │              64 │ expanded_conv_project[0][… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_add (Add)       │ (None, 112, 112, 16)      │               0 │ activation_20[0][0],       │
│                               │                           │                 │ expanded_conv_project_bn[… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_1_expand        │ (None, 112, 112, 64)      │           1,024 │ expanded_conv_add[0][0]    │
│ (Conv2D)                      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_1_expand_bn     │ (None, 112, 112, 64)      │             256 │ expanded_conv_1_expand[0]… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ re_lu_20 (ReLU)               │ (None, 112, 112, 64)      │               

 Total params: 4,674,471 (17.83 MB)

 Trainable params: 3,857,807 (14.72 MB)

 Non-trainable params: 816,664 (3.12 MB)

In [11]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Définir le checkpoint pour sauvegarder les meilleurs poids
checkpoint = ModelCheckpoint('MobileNet_V3_weights.keras',
                             monitor='val_accuracy',
                             verbose=1,
                             mode='max',
                             save_best_only=True)

# Early stopping pour arrêter l'entraînement si la validation stagne
early = EarlyStopping(monitor="val_loss",
                      mode="min",
                      restore_best_weights=True,
                      patience=5)

# Liste des callbacks
callbacks_list = [checkpoint, early]

In [12]:
import time
# Entraînement du modèle tout en mesurant le temps
start_time = time.time()

history = model.fit(
    train_val_ds,
    epochs=15,
    validation_data=val_ds,
    callbacks=callbacks_list,
    verbose=True,
    shuffle=True
)

end_time = time.time()

# Afficher le temps d'entraînement
training_time = end_time - start_time
print(f"Temps d'apprentissage : {training_time} secondes")

# Calcul manuel du F1-score après l'entraînement
precision = history.history['precision'][-1]
recall = history.history['recall'][-1]
f1 = 2 * (precision * recall) / (precision + recall + tf.keras.backend.epsilon()) 
print(f'F1 Score: {f1:.4f}')

Epoch 1/15
291/291 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 0.8950 - nutrient_0_accuracy: 0.5335 - nutrient_0_precision: 0.5867 - nutrient_0_recall: 0.4093

C:\Users\Bamba\AppData\Roaming\Python\Python312\site-packages\keras\src\callbacks\model_checkpoint.py:206: UserWarning: Can save best model only with val_accuracy available, skipping.
  self._save_model(epoch=epoch, batch=None, logs=logs)


291/291 ━━━━━━━━━━━━━━━━━━━━ 426s 1s/step - loss: 0.8941 - nutrient_0_accuracy: 0.5339 - nutrient_0_precision: 0.5870 - nutrient_0_recall: 0.4099 - val_loss: 4.2742 - val_nutrient_0_accuracy: 0.6910 - val_nutrient_0_precision: 0.6922 - val_nutrient_0_recall: 0.6892
Epoch 2/15
291/291 ━━━━━━━━━━━━━━━━━━━━ 398s 1s/step - loss: 0.4095 - nutrient_0_accuracy: 0.7863 - nutrient_0_precision: 0.7916 - nutrient_0_recall: 0.7703 - val_loss: 86.4751 - val_nutrient_0_accuracy: 0.7292 - val_nutrient_0_precision: 0.7292 - val_nutrient_0_recall: 0.7292
Epoch 3/15
291/291 ━━━━━━━━━━━━━━━━━━━━ 398s 1s/step - loss: 0.3656 - nutrient_0_accuracy: 0.8309 - nutrient_0_precision: 0.8364 - nutrient_0_recall: 0.8241 - val_loss: 23.9554 - val_nutrient_0_accuracy: 0.6337 - val_nutrient_0_precision: 0.6337 - val_nutrient_0_recall: 0.6337
Epoch 4/15
291/291 ━━━━━━━━━━━━━━━━━━━━ 398s 1s/step - loss: 0.3741 - nutrient_0_accuracy: 0.8297 - nutrient_0_precision: 0.8321 - nutrient_0_recall: 0.8207 - val_loss: 0.9171 - 

KeyError: 'precision'

In [13]:
model.save("model/MobileNetV3_04_11.keras")

In [14]:
# Évaluation du modèle sur l'ensemble de test
results = model.evaluate(test_ds)

# Affichez les résultats
print(f"Résultats de l'évaluation : {results}")

37/37 ━━━━━━━━━━━━━━━━━━━━ 31s 793ms/step - loss: 0.3755 - nutrient_0_accuracy: 0.8761 - nutrient_0_precision: 0.8759 - nutrient_0_recall: 0.8745
Résultats de l'évaluation : [0.3996272683143616, 0.8696762919425964, 0.8693424463272095, 0.8671209812164307]


In [15]:
# Évaluation sur les données de validation
val_results = model.evaluate(test_ds)

# Affichage de la longueur des résultats et des résultats eux-mêmes
print(f"Nombre total de résultats: {len(val_results)}")
print(f"Résultats de l'évaluation: {val_results}")

# Si vous avez plus ou moins de 13 nutriments, ajustez ici le nombre de nutriments
expected_nutrients = 13

# Vérifiez la structure exacte des résultats pour éviter l'IndexError
if len(val_results) >= expected_nutrients * 4 + 1:
    total_val_loss = val_results[0]
    nutrient_losses = val_results[1:expected_nutrients + 1]
    nutrient_accuracies = val_results[expected_nutrients + 1:expected_nutrients * 2 + 1]
    nutrient_precisions = val_results[expected_nutrients * 2 + 1:expected_nutrients * 3 + 1]
    nutrient_recalls = val_results[expected_nutrients * 3 + 1:expected_nutrients * 4 + 1]

    # Affichage des résultats globaux
    print(f"Total Validation Loss: {total_val_loss}")

    # Affichage des résultats pour chaque nutriment
    for i in range(expected_nutrients):
        print(f"Nutrient {i+1}:")
        print(f"  Validation Loss: {nutrient_losses[i]}")
        print(f"  Validation Accuracy: {nutrient_accuracies[i]}")
        print(f"  Precision: {nutrient_precisions[i]}")
        print(f"  Recall: {nutrient_recalls[i]}")
else:
    print("La structure des résultats est différente de celle attendue.")

37/37 ━━━━━━━━━━━━━━━━━━━━ 31s 793ms/step - loss: 0.3399 - nutrient_0_accuracy: 0.8831 - nutrient_0_precision: 0.8828 - nutrient_0_recall: 0.8808
Nombre total de résultats: 4
Résultats de l'évaluation: [0.3913249969482422, 0.8696762919425964, 0.8693424463272095, 0.8671209812164307]
La structure des résultats est différente de celle attendue.


In [16]:
# Évaluation du modèle sur l'ensemble de test
results = model.evaluate(test_ds_mixte)

# Affichez les résultats
print(f"Résultats de l'évaluation : {results}")

37/37 ━━━━━━━━━━━━━━━━━━━━ 30s 775ms/step - loss: 1.8557 - nutrient_0_accuracy: 0.8088 - nutrient_0_precision: 0.8089 - nutrient_0_recall: 0.8065
Résultats de l'évaluation : [1.707862377166748, 0.8200691938400269, 0.8206239342689514, 0.8192041516304016]


In [17]:
# Évaluation sur les données de validation
val_results = model.evaluate(test_ds_mixte)

# Affichage de la longueur des résultats et des résultats eux-mêmes
print(f"Nombre total de résultats: {len(val_results)}")
print(f"Résultats de l'évaluation: {val_results}")

# Si vous avez plus ou moins de 13 nutriments, ajustez ici le nombre de nutriments
expected_nutrients = 13

# Vérifiez la structure exacte des résultats pour éviter l'IndexError
if len(val_results) >= expected_nutrients * 4 + 1:
    total_val_loss = val_results[0]
    nutrient_losses = val_results[1:expected_nutrients + 1]
    nutrient_accuracies = val_results[expected_nutrients + 1:expected_nutrients * 2 + 1]
    nutrient_precisions = val_results[expected_nutrients * 2 + 1:expected_nutrients * 3 + 1]
    nutrient_recalls = val_results[expected_nutrients * 3 + 1:expected_nutrients * 4 + 1]

    # Affichage des résultats globaux
    print(f"Total Validation Loss: {total_val_loss}")

    # Affichage des résultats pour chaque nutriment
    for i in range(expected_nutrients):
        print(f"Nutrient {i+1}:")
        print(f"  Validation Loss: {nutrient_losses[i]}")
        print(f"  Validation Accuracy: {nutrient_accuracies[i]}")
        print(f"  Precision: {nutrient_precisions[i]}")
        print(f"  Recall: {nutrient_recalls[i]}")
else:
    print("La structure des résultats est différente de celle attendue.")

37/37 ━━━━━━━━━━━━━━━━━━━━ 31s 794ms/step - loss: 1.5509 - nutrient_0_accuracy: 0.8066 - nutrient_0_precision: 0.8070 - nutrient_0_recall: 0.8048
Nombre total de résultats: 4
Résultats de l'évaluation: [1.7535747289657593, 0.8174740672111511, 0.8180242776870728, 0.8166090250015259]
La structure des résultats est différente de celle attendue.


In [18]:
# Évaluation du modèle sur l'ensemble de test
results = model.evaluate(test_ds_maiis)

# Affichez les résultats
print(f"Résultats de l'évaluation : {results}")

37/37 ━━━━━━━━━━━━━━━━━━━━ 30s 769ms/step - loss: 1.7899 - nutrient_0_accuracy: 0.8052 - nutrient_0_precision: 0.8062 - nutrient_0_recall: 0.8052
Résultats de l'évaluation : [1.7170300483703613, 0.8183391094207764, 0.8190476298332214, 0.8183391094207764]


In [19]:
# Évaluation sur les données de validation
val_results = model.evaluate(test_ds_maiis)

# Affichage de la longueur des résultats et des résultats eux-mêmes
print(f"Nombre total de résultats: {len(val_results)}")
print(f"Résultats de l'évaluation: {val_results}")

# Si vous avez plus ou moins de 13 nutriments, ajustez ici le nombre de nutriments
expected_nutrients = 13

# Vérifiez la structure exacte des résultats pour éviter l'IndexError
if len(val_results) >= expected_nutrients * 4 + 1:
    total_val_loss = val_results[0]
    nutrient_losses = val_results[1:expected_nutrients + 1]
    nutrient_accuracies = val_results[expected_nutrients + 1:expected_nutrients * 2 + 1]
    nutrient_precisions = val_results[expected_nutrients * 2 + 1:expected_nutrients * 3 + 1]
    nutrient_recalls = val_results[expected_nutrients * 3 + 1:expected_nutrients * 4 + 1]

    # Affichage des résultats globaux
    print(f"Total Validation Loss: {total_val_loss}")

    # Affichage des résultats pour chaque nutriment
    for i in range(expected_nutrients):
        print(f"Nutrient {i+1}:")
        print(f"  Validation Loss: {nutrient_losses[i]}")
        print(f"  Validation Accuracy: {nutrient_accuracies[i]}")
        print(f"  Precision: {nutrient_precisions[i]}")
        print(f"  Recall: {nutrient_recalls[i]}")
else:
    print("La structure des résultats est différente de celle attendue.")

37/37 ━━━━━━━━━━━━━━━━━━━━ 30s 760ms/step - loss: 1.7979 - nutrient_0_accuracy: 0.7979 - nutrient_0_precision: 0.7982 - nutrient_0_recall: 0.7979
Nombre total de résultats: 4
Résultats de l'évaluation: [1.780985951423645, 0.8122837543487549, 0.8129870295524597, 0.8122837543487549]
La structure des résultats est différente de celle attendue.


In [20]:
# Évaluation sur les données de validation
val_results = model.evaluate(val_ds)

# Affichage de la longueur des résultats et des résultats eux-mêmes
print(f"Nombre total de résultats: {len(val_results)}")
print(f"Résultats de l'évaluation: {val_results}")

# Si vous avez plus ou moins de 13 nutriments, ajustez ici le nombre de nutriments
expected_nutrients = 13

# Vérifiez la structure exacte des résultats pour éviter l'IndexError
if len(val_results) >= expected_nutrients * 4 + 1:
    total_val_loss = val_results[0]
    nutrient_losses = val_results[1:expected_nutrients + 1]
    nutrient_accuracies = val_results[expected_nutrients + 1:expected_nutrients * 2 + 1]
    nutrient_precisions = val_results[expected_nutrients * 2 + 1:expected_nutrients * 3 + 1]
    nutrient_recalls = val_results[expected_nutrients * 3 + 1:expected_nutrients * 4 + 1]

    # Affichage des résultats globaux
    print(f"Total Validation Loss: {total_val_loss}")

    # Affichage des résultats pour chaque nutriment
    for i in range(expected_nutrients):
        print(f"Nutrient {i+1}:")
        print(f"  Validation Loss: {nutrient_losses[i]}")
        print(f"  Validation Accuracy: {nutrient_accuracies[i]}")
        print(f"  Precision: {nutrient_precisions[i]}")
        print(f"  Recall: {nutrient_recalls[i]}")
else:
    print("La structure des résultats est différente de celle attendue.")

36/36 ━━━━━━━━━━━━━━━━━━━━ 29s 796ms/step - loss: 0.3738 - nutrient_0_accuracy: 0.8517 - nutrient_0_precision: 0.8521 - nutrient_0_recall: 0.8502
Nombre total de résultats: 4
Résultats de l'évaluation: [0.3682060241699219, 0.8489583134651184, 0.8495652079582214, 0.8480902910232544]
La structure des résultats est différente de celle attendue.


In [21]:
# Évaluation sur les données de validation
val_results = model.evaluate(val_ds)

# Affichage de la longueur des résultats et des résultats eux-mêmes
print(f"Nombre total de résultats: {len(val_results)}")
print(f"Résultats de l'évaluation: {val_results}")

# Si vous avez plus ou moins de 13 nutriments, ajustez ici le nombre de nutriments
expected_nutrients = 13

# Vérifiez la structure exacte des résultats pour éviter l'IndexError
if len(val_results) >= expected_nutrients * 4 + 1:
    total_val_loss = val_results[0]
    nutrient_losses = val_results[1:expected_nutrients + 1]
    nutrient_accuracies = val_results[expected_nutrients + 1:expected_nutrients * 2 + 1]
    nutrient_precisions = val_results[expected_nutrients * 2 + 1:expected_nutrients * 3 + 1]
    nutrient_recalls = val_results[expected_nutrients * 3 + 1:expected_nutrients * 4 + 1]

    # Affichage des résultats globaux
    print(f"Total Validation Loss: {total_val_loss}")

    # Affichage des résultats pour chaque nutriment
    for i in range(expected_nutrients):
        print(f"Nutrient {i+1}:")
        print(f"  Validation Loss: {nutrient_losses[i]}")
        print(f"  Validation Accuracy: {nutrient_accuracies[i]}")
        print(f"  Precision: {nutrient_precisions[i]}")
        print(f"  Recall: {nutrient_recalls[i]}")
else:
    print("La structure des résultats est différente de celle attendue.")

36/36 ━━━━━━━━━━━━━━━━━━━━ 29s 800ms/step - loss: 0.3819 - nutrient_0_accuracy: 0.8481 - nutrient_0_precision: 0.8480 - nutrient_0_recall: 0.8454
Nombre total de résultats: 4
Résultats de l'évaluation: [0.37357333302497864, 0.8454861044883728, 0.8460869789123535, 0.8446180820465088]
La structure des résultats est différente de celle attendue.


In [22]:
import time
from tensorflow.keras import backend as K

# Fonction pour calculer la moyenne d'une métrique
def mean_metric(metric_values):
    return sum(metric_values) / len(metric_values)

# Nombre de nutriments
num_nutrients = 13

# Fonction pour calculer les moyennes pour chaque nutriment
def mean_nutrient_metric(metric_name, history):
    metrics = []
    for nutrient in range(num_nutrients):
        key = f'nutrient_{nutrient}_{metric_name}'
        if key in history:
            metrics.append(history[key])
    return [mean_metric(metric) for metric in zip(*metrics)]  # Moyenne sur les époques

# Calcul des moyennes pour l'ensemble des époques (entraînement)
mean_train_loss = mean_metric(history.history['loss'])
mean_train_accuracy = mean_nutrient_metric('accuracy', history.history)
mean_train_precision = mean_nutrient_metric('precision', history.history)
mean_train_recall = mean_nutrient_metric('recall', history.history)

# Calcul du F1-score moyen pour l'ensemble des époques (entraînement)
f1_train_scores = [2 * (p * r) / (p + r + K.epsilon()) 
                   for p, r in zip(mean_train_precision, mean_train_recall)]
mean_f1_train = mean_metric(f1_train_scores)

# Calcul des moyennes pour l'ensemble des époques (validation)
mean_val_loss = mean_metric(history.history['val_loss'])
mean_val_accuracy = mean_nutrient_metric('accuracy', history.history)
mean_val_precision = mean_nutrient_metric('precision', history.history)
mean_val_recall = mean_nutrient_metric('recall', history.history)

# Calcul du F1-score moyen pour l'ensemble des époques (validation)
f1_val_scores = [2 * (p * r) / (p + r + K.epsilon()) 
                 for p, r in zip(mean_val_precision, mean_val_recall)]
mean_f1_val = mean_metric(f1_val_scores)

# Affichage des résultats
print(f"Moyenne de la perte sur l'ensemble d'entraînement : {mean_train_loss:.4f}")
print(f"Moyenne de l'accuracy sur l'ensemble d'entraînement : {mean_metric(mean_train_accuracy):.4f}")
print(f"Moyenne de la précision sur l'ensemble d'entraînement : {mean_metric(mean_train_precision):.4f}")
print(f"Moyenne du rappel sur l'ensemble d'entraînement : {mean_metric(mean_train_recall):.4f}")
print(f"Moyenne du F1-score sur l'ensemble d'entraînement : {mean_f1_train:.4f}")

print(f"Moyenne de la perte sur l'ensemble de validation : {mean_val_loss:.4f}")
print(f"Moyenne de l'accuracy sur l'ensemble de validation : {mean_metric(mean_val_accuracy):.4f}")
print(f"Moyenne de la précision sur l'ensemble de validation : {mean_metric(mean_val_precision):.4f}")
print(f"Moyenne du rappel sur l'ensemble de validation : {mean_metric(mean_val_recall):.4f}")
print(f"Moyenne du F1-score sur l'ensemble de validation : {mean_f1_val:.4f}")

Moyenne de la perte sur l'ensemble d'entraînement : 0.2708
Moyenne de l'accuracy sur l'ensemble d'entraînement : 0.8708
Moyenne de la précision sur l'ensemble d'entraînement : 0.8769
Moyenne du rappel sur l'ensemble d'entraînement : 0.8613
Moyenne du F1-score sur l'ensemble d'entraînement : 0.8686
Moyenne de la perte sur l'ensemble de validation : 9.2549
Moyenne de l'accuracy sur l'ensemble de validation : 0.8708
Moyenne de la précision sur l'ensemble de validation : 0.8769
Moyenne du rappel sur l'ensemble de validation : 0.8613
Moyenne du F1-score sur l'ensemble de validation : 0.8686
